# Get crops from Xenium cell_boundaries.parquet files as .tif files. 

In [ ]:
#read in reference files
import pandas as pd
from pathlib import Path
import numpy as np

#references for coordinates of initial section crops
##This is a single csv that combines the info from each of the bounding box csv files in each Xenium output folder

coord_reference = pd.read_csv("/path/to/coords.csv")
coord_reference['BX'] = coord_reference['BX'].astype(np.int64)
coord_reference['BY'] = coord_reference['BY'].astype(np.int64)

#cell boundary files from each xenium point
Ctrl_boundaries = Path("/path/to/cell_boundaries.parquet")
D3_boundaries = Path("/path/to/cell_boundaries1.parquet")
D7_boundaries = Path("/path/to/cell_boundaries2.parquet")
D10_boundaries = Path("/path/to/cell_boundaries3.parquet")

#folder(s) with all your crops 
test_folder = Path("path/to/folder/with/crops")

In [5]:
#Extract coordinate information and image names from crop names and enter them into a dataframe
#crop name format = crop'x'_condition_section_X'num'Y'num'.tif, where 'x' is the crop number, 'condition' is Ctrl/D3/D7/D10, section is F1/M2 etc, and 'num' are the x/y coords. 

import re

crop_nums = []
conditions = []
sections = []
local_x_coords = []
local_y_coords = []
for img in test_folder.glob("crop*_masks.tif"):
    stem = img.stem
    parts = stem.split("_")

    crop_num = parts[0].replace("crop", "")
    crop_nums.append(crop_num)

    condition = parts[1]
    conditions.append(condition)

    section = parts[2]
    sections.append(section)

    coords_part = parts[3]
    x = re.search(r'X(\d+)', coords_part).group(1)
    local_x_coords.append(x)

    y = re.search(r'Y(\d+)', coords_part).group(1)
    local_y_coords.append(y)

crop_info = pd.DataFrame({'Crop_Num':crop_nums, 'Condition':conditions, 'Section':sections, 'X':local_x_coords, 'Y':local_y_coords})
crop_info.head()

,Crop_Num,Condition,Section,X,Y
0,0,D10,F2,12727,2147
1,2,D10,M1,10880,1113
2,4,D10,M1,20508,1299
3,10,D10,M2,51,1731
4,2,D7,M1,11309,990


In [6]:
#Using the crop origin info that you just extracted and the coordinate reference csv, insert columns that detail the (x,y) offsets for the entire xenium slide
global_x_offset = []
global_y_offset = []

for _, row in crop_info.iterrows(): #loop through the rows of the dataframe
    #search for the row in coord_reference which has the same values in the 'Condition' and 'Section' 
    ref_row = coord_reference[
        (coord_reference['Condition'] == row['Condition']) &
        (coord_reference['Section'] == row['Section'])
    ].iloc[0]

    global_x = ref_row['BX'] + int(row['X'])
    global_y = ref_row['BY'] + int(row['Y'])

    global_x_offset.append(global_x)
    global_y_offset.append(global_y)

crop_info['global_x_offset'] = global_x_offset
crop_info['global_y_offset'] = global_y_offset


In [7]:
#defining functions to get a tif crop from a xenium cell_boundaries file based on x,y offsets and cell_boundaries file
from __future__ import annotations
import numpy as np
import pandas as pd
import tifffile as tif
from skimage.draw import polygon


# Xenium pixel size
PIXEL_SIZE_UM = 0.2125

# Output window size in pixels
ROI_SIZE = 750

def rasterize_cell_boundaries_to_mask(
    df: pd.DataFrame,
    x_offset_px: int,
    y_offset_px: int,
    roi_size: int,
    pixel_size_um: float,
) -> np.ndarray:
    """
    Convert cell boundary polygons to a label mask for one ROI.

    Assumptions:
    - vertex_x and vertex_y are in micrometers
    - x_offset_px and y_offset_px are pixel coordinates from the top-left of the full image
    - rows belonging to the same cell_id are already in boundary order
    """

    required_cols = {"cell_id", "vertex_x", "vertex_y", "label_id"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    mask = np.zeros((roi_size, roi_size), dtype=np.uint32)

    # ROI bounds in full-image pixel coordinates
    x0 = x_offset_px
    y0 = y_offset_px
    x1 = x0 + roi_size
    y1 = y0 + roi_size

    # Convert all coordinates from micrometers to full-image pixel coordinates
    df = df.copy()
    df["x_px"] = df["vertex_x"] / pixel_size_um
    df["y_px"] = df["vertex_y"] / pixel_size_um

    # Group by cell and draw each polygon into the ROI mask
    for cell_id, g in df.groupby("cell_id", sort=False):
        label_id = int(g["label_id"].iloc[0])
        if label_id == 0:
            continue

        # Polygon vertices in ROI-local pixel coordinates
        x = g["x_px"].to_numpy() - x0
        y = g["y_px"].to_numpy() - y0

        # Quick bbox rejection
        if x.max() < 0 or x.min() >= roi_size or y.max() < 0 or y.min() >= roi_size:
            continue

        # Rasterize polygon into ROI
        rr, cc = polygon(y, x, shape=mask.shape)
        mask[rr, cc] = label_id

    return mask


In [ ]:
output_dir = Path("/location/to/save/xen_crops")

for _, row in crop_info.iterrows():
    crop_num = row['Crop_Num']
    section = row['Section']
    condition = row['Condition']
    if condition == 'Ctrl':
        cell_boundary_file = Ctrl_boundaries
    elif condition == 'D3':
        cell_boundary_file = D3_boundaries
    elif condition == 'D7':
        cell_boundary_file = D7_boundaries
    elif condition == 'D10':
        cell_boundary_file = D10_boundaries

    X_OFFSET_PX = row['global_x_offset']
    Y_OFFSET_PX = row['global_y_offset']
    
    output_path = output_dir.joinpath(f'crop{crop_num}{condition}_{section}_xen.tif')
    
    df = pd.read_parquet(cell_boundary_file)
    roi_mask = rasterize_cell_boundaries_to_mask(
        df=df,
        x_offset_px=X_OFFSET_PX,
        y_offset_px=Y_OFFSET_PX,
        roi_size=ROI_SIZE,
        pixel_size_um=PIXEL_SIZE_UM,
    )

    tif.imwrite(output_path, roi_mask)

    print(f'saved {output_path}')

saved D:\Dom\Psoriasis project\4th year data\cellpose training\Crops_Round2\Final Choices\V7\Xen test crops\crop0D10_F2_xen.tif
saved D:\Dom\Psoriasis project\4th year data\cellpose training\Crops_Round2\Final Choices\V7\Xen test crops\crop2D10_M1_xen.tif
saved D:\Dom\Psoriasis project\4th year data\cellpose training\Crops_Round2\Final Choices\V7\Xen test crops\crop4D10_M1_xen.tif
saved D:\Dom\Psoriasis project\4th year data\cellpose training\Crops_Round2\Final Choices\V7\Xen test crops\crop10D10_M2_xen.tif
saved D:\Dom\Psoriasis project\4th year data\cellpose training\Crops_Round2\Final Choices\V7\Xen test crops\crop2D7_M1_xen.tif
saved D:\Dom\Psoriasis project\4th year data\cellpose training\Crops_Round2\Final Choices\V7\Xen test crops\crop6D7_F1_xen.tif
saved D:\Dom\Psoriasis project\4th year data\cellpose training\Crops_Round2\Final Choices\V7\Xen test crops\crop9D7_M2_xen.tif
saved D:\Dom\Psoriasis project\4th year data\cellpose training\Crops_Round2\Final Choices\V7\Xen test crop